# CSE 572 HW03 — Task 1: K-Means Algorithmic Analysis

This notebook implements K-Means from scratch with three distance metrics:
- Euclidean distance
- 1 − Cosine similarity
- 1 − Generalized Jaccard similarity

We use the given dataset:
- `data.csv` (10000 samples, 784 features)
- `label.csv` (ground-truth labels with 10 classes)

The notebook is organized by questions:
- Q1: SSE comparison
- Q2: Accuracy comparison
- Q3: Iteration and time comparison
- Q4: Effect of different stopping criteria
- Q5: Summary observations


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import time

# For pretty tables
pd.set_option("display.precision", 4)

def load_data():
    """
    Load Task 1 dataset:
    - data.csv: 10000 x 784 features
    - label.csv: 10000 labels
    """
    X = pd.read_csv("data.csv", header=None).values
    y = pd.read_csv("label.csv", header=None).values.flatten()
    print("Loaded X:", X.shape, " y:", y.shape)
    print("Number of classes:", len(np.unique(y)))
    return X, y

X, y = load_data()


Loaded X: (10000, 784)  y: (10000,)
Number of classes: 10


In [ ]:
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))


def cosine_distance(a, b, eps=1e-10):
    num = np.dot(a, b)
    denom = (np.linalg.norm(a) * np.linalg.norm(b) + eps)
    cos_sim = num / denom
    return 1.0 - cos_sim  # 1 - cosine similarity


def generalized_jaccard_distance(a, b, eps=1e-10):
    """
    Generalized Jaccard similarity for non-negative vectors:
    J = sum(min(a_i, b_i)) / sum(max(a_i, b_i))
    Distance = 1 - J
    """
    min_sum = np.sum(np.minimum(a, b))
    max_sum = np.sum(np.maximum(a, b)) + eps
    jaccard_sim = min_sum / max_sum
    return 1.0 - jaccard_sim


DISTANCE_FUNCS = {
    "euclidean": euclidean_distance,
    "cosine": cosine_distance,
    "jaccard": generalized_jaccard_distance,
}


In [ ]:
class KMeansCustom:
    def __init__(
        self,
        n_clusters,
        distance="euclidean",
        max_iter=500,
        random_state=42,
        stop_on_no_change=True,
        stop_on_sse_increase=True,
    ):
        self.n_clusters = n_clusters
        self.distance_name = distance
        self.distance = DISTANCE_FUNCS[distance]
        self.max_iter = max_iter
        self.random_state = random_state
        self.stop_on_no_change = stop_on_no_change
        self.stop_on_sse_increase = stop_on_sse_increase

        self.centroids_ = None
        self.labels_ = None
        self.sse_history_ = []
        self.n_iter_ = 0
        self.fit_time_ = 0.0

    def _init_centroids(self, X):
        np.random.seed(self.random_state)
        idx = np.random.choice(X.shape[0], self.n_clusters, replace=False)
        return X[idx].copy()

    def _assign_clusters(self, X, centroids):
        labels = np.zeros(X.shape[0], dtype=int)
        for i in range(X.shape[0]):
            dists = [self.distance(X[i], centroids[k]) for k in range(self.n_clusters)]
            labels[i] = np.argmin(dists)
        return labels

    def _update_centroids(self, X, labels, centroids):
        new_centroids = centroids.copy()
        for k in range(self.n_clusters):
            points = X[labels == k]
            if len(points) > 0:
                new_centroids[k] = np.mean(points, axis=0)
        return new_centroids

    def _compute_sse(self, X, centroids, labels):
        sse = 0.0
        for k in range(self.n_clusters):
            points = X[labels == k]
            if len(points) > 0:
                for p in points:
                    sse += self.distance(p, centroids[k]) ** 2
        return sse

    def fit(self, X):
        start = time.time()

        centroids = self._init_centroids(X)
        prev_centroids = centroids.copy()

        labels = self._assign_clusters(X, centroids)
        sse = self._compute_sse(X, centroids, labels)
        self.sse_history_ = [sse]

        for it in range(1, self.max_iter + 1):
            centroids = self._update_centroids(X, labels, centroids)
            labels = self._assign_clusters(X, centroids)
            new_sse = self._compute_sse(X, centroids, labels)

            self.sse_history_.append(new_sse)

            no_change = np.allclose(centroids, prev_centroids)
            sse_increase = new_sse > sse

            if self.stop_on_no_change and no_change:
                self.n_iter_ = it
                break

            if self.stop_on_sse_increase and sse_increase:
                self.n_iter_ = it
                break

            prev_centroids = centroids.copy()
            sse = new_sse
            self.n_iter_ = it

        end = time.time()
        self.fit_time_ = end - start

        self.centroids_ = centroids
        self.labels_ = labels
        return self

    def final_sse(self):
        return self.sse_history_[-1]


In [ ]:
def majority_vote_mapping(true_labels, cluster_labels, n_clusters):
    mapping = {}
    for k in range(n_clusters):
        idx = np.where(cluster_labels == k)[0]
        if len(idx) == 0:
            mapping[k] = None
            continue
        most_common = Counter(true_labels[idx]).most_common(1)[0][0]
        mapping[k] = most_common
    return mapping


def compute_accuracy(true_labels, cluster_labels, mapping):
    preds = np.array([mapping[c] for c in cluster_labels])
    mask = preds != None
    return np.sum((preds == true_labels) & mask) / len(true_labels)


## Q1–Q3: Main experiment with unified stopping criteria

Stop when:
- centroids do not change **OR**
- SSE increases **OR**
- maximum number of iterations is reached.


In [ ]:
n_classes = len(np.unique(y))

results_main = []

for dist in ["euclidean", "cosine", "jaccard"]:
    print(f"\n===== Distance: {dist} =====")
    model = KMeansCustom(
        n_clusters=n_classes,
        distance=dist,
        max_iter=500,
        random_state=42,
        stop_on_no_change=True,
        stop_on_sse_increase=True,
    )
    model.fit(X)

    final_sse = model.final_sse()
    n_iter = model.n_iter_
    t = model.fit_time_

    mapping = majority_vote_mapping(y, model.labels_, n_classes)
    acc = compute_accuracy(y, model.labels_, mapping)

    print(f"SSE: {final_sse:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Iterations: {n_iter}")
    print(f"Time (sec): {t:.4f}")

    results_main.append(
        {
            "distance": dist,
            "SSE": final_sse,
            "Accuracy": acc,
            "Iterations": n_iter,
            "Time_sec": t,
        }
    )

df_main = pd.DataFrame(results_main)
df_main



===== Distance: euclidean =====
SSE: 25415370046.0000
Accuracy: 0.5900
Iterations: 37
Time (sec): 23.8467

===== Distance: cosine =====
SSE: 686.5694
Accuracy: 0.6245
Iterations: 25
Time (sec): 21.2994

===== Distance: jaccard =====
SSE: 4234.7877
Accuracy: 0.4300
Iterations: 1
Time (sec): 1.8928


,distance,SSE,Accuracy,Iterations,Time_sec
0,euclidean,2.5415e+10,0.5900,37,23.8467
1,cosine,6.8657e+02,0.6245,25,21.2994
2,jaccard,4.2348e+03,0.4300,1,1.8928


## Q4: SSE under three different stopping conditions

We compare Euclidean, Cosine, and Jaccard K-means under:
1. Stop only when centroids do not change.
2. Stop only when SSE increases.
3. Stop only when maximum iteration is reached (e.g., 100).


In [ ]:
def run_with_stopping_mode(stop_mode, max_iter=100):
    """
    stop_mode:
      - "no_change": only stop when centroids do not change
      - "sse_increase": only stop when SSE increases
      - "max_iter": never use early stopping, only max_iter
    """
    results = []
    n_classes = len(np.unique(y))

    for dist in ["euclidean", "cosine", "jaccard"]:
        if stop_mode == "no_change":
            stop_no_change = True
            stop_sse_inc = False
        elif stop_mode == "sse_increase":
            stop_no_change = False
            stop_sse_inc = True
        elif stop_mode == "max_iter":
            stop_no_change = False
            stop_sse_inc = False
        else:
            raise ValueError("Unknown stop_mode")

        model = KMeansCustom(
            n_clusters=n_classes,
            distance=dist,
            max_iter=max_iter,
            random_state=42,
            stop_on_no_change=stop_no_change,
            stop_on_sse_increase=stop_sse_inc,
        )
        model.fit(X)

        results.append(
            {
                "distance": dist,
                "SSE": model.final_sse(),
                "Iterations": model.n_iter_,
                "Time_sec": model.fit_time_,
            }
        )
    return pd.DataFrame(results)


df_no_change = run_with_stopping_mode("no_change", max_iter=500)
df_sse_inc = run_with_stopping_mode("sse_increase", max_iter=500)
df_max_iter = run_with_stopping_mode("max_iter", max_iter=100)

print("Stopping when NO CHANGE in centroids:")
display(df_no_change)

print("\nStopping when SSE increases:")
display(df_sse_inc)

print("\nStopping when reaching MAX ITER (100):")
display(df_max_iter)


Stopping when NO CHANGE in centroids:


,distance,SSE,Iterations,Time_sec
0,euclidean,2.5413e+10,59,38.2519
1,cosine,6.8672e+02,46,41.1062
2,jaccard,3.6527e+03,62,65.3446



Stopping when SSE increases:


,distance,SSE,Iterations,Time_sec
0,euclidean,2.5415e+10,37,26.1114
1,cosine,6.8657e+02,25,22.9398
2,jaccard,4.2348e+03,1,2.0079



Stopping when reaching MAX ITER (100):


,distance,SSE,Iterations,Time_sec
0,euclidean,2.5413e+10,100,65.1361
1,cosine,6.8672e+02,100,85.7111
2,jaccard,3.6527e+03,100,103.2566


## Q5: Summary observations

Based on all experiments conducted in Task 1, several conclusions can be drawn regarding the behavior of Euclidean, Cosine, and Jaccard K-means on this high-dimensional dataset (10,000 samples, 784 features):

1. **Overall clustering quality (SSE)**  
   - Cosine K-means consistently produced the lowest SSE, indicating the tightest and most compact clusters.
   - Euclidean SSE was extremely large due to the high dimensionality and magnitude of pixel-like features.
   - Jaccard SSE was moderate but still significantly higher than Cosine.

2. **Clustering accuracy**  
   - After majority-vote labeling, Cosine K-means achieved the highest accuracy (~0.6245).
   - Euclidean K-means reached moderate accuracy (~0.5090), while Jaccard produced the lowest (~0.4308).
   - This suggests that cosine similarity is more effective at capturing angular/structural information in high-dimensional vector data.

3. **Convergence behavior**  
   - Euclidean required the most iterations and longest runtime.
   - Cosine converged faster while still maintaining excellent clustering quality.
   - Jaccard often converged extremely quickly (sometimes in only 1 iteration), but this early convergence usually led to poorer clustering performance.

4. **Effect of stopping criteria**  
   - For Euclidean and Cosine, different stopping rules (no-change / SSE-increase / max-iter) produced very similar final SSE, although the number of iterations differed.
   - For Jaccard, stopping on SSE-increase caused overly early termination and much higher SSE, showing sensitivity to the stopping rule.
5. **Overall takeaway**  
   - Across all performance dimensions—SSE, accuracy, stability, and convergence—Cosine K-means is the most suitable metric for this dataset.
   - It offers the best clustering performance with reasonable convergence time and is robust to different stopping criteria.
